In [0]:
ruta_raw = "/Volumes/workspace/raw/kaggle_adidas_us_sales"
archivo_fuente = "Adidas US Sales*.csv"
checkpoint = "/Volumes/workspace/bronze/checkpoint"
bronze = "workspace.bronze.adidas_us_sales"

In [0]:
esquema = """
    sales_id BIGINT, retailer STRING, retailer_id BIGINT, invoice_date DATE,
    region STRING, state STRING, city STRING, product STRING,
    price_per_unit INT, units_sold INT, total_sales INT, operating_profit INT,
    sales_method STRING
"""

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {bronze} ({esquema}) USING DELTA
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

In [0]:
propiedades = {fila["key"]: fila["value"] for fila in spark.sql(f"SHOW TBLPROPERTIES {bronze}").collect()}
if propiedades.get("delta.enableChangeDataFeed", "false").lower() != "true":
    spark.sql(f"ALTER TABLE {bronze} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")


In [0]:
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("dateFormat", "yyyy-MM-dd")
    .option("mode", "FAILFAST")
    .option("pathGlobFilter", archivo_fuente)
    .schema(esquema)
    .load(ruta_raw)
)
flujo = (
    df.writeStream.format("delta").option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(bronze)
)
flujo.awaitTermination()
